In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
path = "/content/drive/MyDrive/06 - Green Washing AI/analysis/"

In [14]:
# all 5 prompt types, all 3 conditions, and all 4 companies are guaranteed to be represented.

import pandas as pd
import numpy as np

RANDOM_SEED = 127
rng = np.random.default_rng(RANDOM_SEED)

# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_csv(path + "generated_content/evaluated_greenwashing_results.csv")  # change sep if needed

#  ── Config ────────────────────────────────────────────────────────────────────
TARGET_N      = 20
SCORE_COL     = "total_greenwashing_score"
CONDITION_COL = "grounding_condition"
COMPANY_COL   = "company_name"
PROMPT_COL    = "prompt_type"

CONDITIONS   = df[CONDITION_COL].unique().tolist()
COMPANIES    = df[COMPANY_COL].unique().tolist()
PROMPT_TYPES = df[PROMPT_COL].unique().tolist()

In [15]:
def random_pick(pool_df, n, exclude_ids):
    """Randomly pick n rows from pool_df excluding already selected ids."""
    candidates = pool_df[~pool_df["run_id"].isin(exclude_ids)]
    n = min(n, len(candidates))
    if n == 0:
        return []
    idx = rng.choice(len(candidates), size=n, replace=False)
    return candidates.iloc[idx]["run_id"].tolist()

selected_ids = set()

# STEP 1 — Randomly pick exactly 1 output per company × prompt type cell
# 4 companies × 5 prompt types = 20 cells = 20 outputs
# Each company gets all 5 prompt types; each prompt type appears exactly 4 times.
for co in COMPANIES:
    for pt in PROMPT_TYPES:
        cell = df[(df[COMPANY_COL] == co) & (df[PROMPT_COL] == pt)]
        picked = random_pick(cell, 1, selected_ids)
        selected_ids.update(picked)

In [16]:
# #  STEP 1 — Randomly pick 1 output per prompt type (5 outputs)
# for pt in PROMPT_TYPES:
#     subset = df[df[PROMPT_COL] == pt]
#     picked = random_pick(subset, 1, selected_ids)
#     selected_ids.update(picked)

# # STEP 2 — Ensure all 3 conditions have at least 2 outputs (random fill)
# for cond in CONDITIONS:
#     already = df[df["run_id"].isin(selected_ids) & (df[CONDITION_COL] == cond)]
#     needed = max(0, 2 - len(already))
#     if needed > 0:
#         pool = df[df[CONDITION_COL] == cond]
#         picked = random_pick(pool, needed, selected_ids)
#         selected_ids.update(picked)

# # STEP 3 — Ensure all 4 companies have at least 2 outputs (random fill)
# for co in COMPANIES:
#     already = df[df["run_id"].isin(selected_ids) & (df[COMPANY_COL] == co)]
#     needed = max(0, 2 - len(already))
#     if needed > 0:
#         pool = df[df[COMPANY_COL] == co]
#         picked = random_pick(pool, needed, selected_ids)
#         selected_ids.update(picked)

# # STEP 4 — Fill remaining slots randomly from the full dataset
# remaining = TARGET_N - len(selected_ids)
# if remaining > 0:
#     picked = random_pick(df, remaining, selected_ids)
#     selected_ids.update(picked)

In [17]:
#  ── Build output ──────────────────────────────────────────────────────────────
anchor_set = df[df["run_id"].isin(selected_ids)].copy()
anchor_set = anchor_set.sort_values([COMPANY_COL, CONDITION_COL, PROMPT_COL])

SCORE_DIMS = ["score_vagueness", "score_misleading", "score_concealment",
              "score_overselling", "score_irrelevance"]

output_cols = [
    "run_id", COMPANY_COL, CONDITION_COL, PROMPT_COL,
    SCORE_COL, *SCORE_DIMS,
    "sustainability_focus", "generated_text",
    "justification_vagueness", "justification_misleading",
    "justification_concealment", "justification_overselling", "justification_irrelevance"
]
anchor_set[output_cols].to_csv(path + f"anchor_set_20_{RANDOM_SEED}.csv", index=False)

In [18]:
#  ── Summary diagnostics ───────────────────────────────────────────────────────
print(f"\n{'='*55}")
print(f"  ANCHOR SET — {len(anchor_set)} outputs selected  (seed={RANDOM_SEED})")
print(f"{'='*55}")

print("\n▸ Score distribution:")
print(anchor_set[SCORE_COL].describe().to_string())

print(f"\n  Min score : {anchor_set[SCORE_COL].min()}")
print(f"  Max score : {anchor_set[SCORE_COL].max()}")
print(f"  Mean score: {anchor_set[SCORE_COL].mean():.2f}")

print("\n▸ Grounding condition coverage:")
print(anchor_set[CONDITION_COL].value_counts().to_string())

print("\n▸ Company coverage:")
print(anchor_set[COMPANY_COL].value_counts().to_string())

print("\n▸ Prompt type coverage:")
print(anchor_set[PROMPT_COL].value_counts().to_string())

print("\n▸ Sustainability focus split:")
print(anchor_set["sustainability_focus"].value_counts().to_string())

print("\n▸ Selected run IDs:")
for rid in sorted(anchor_set["run_id"].tolist()):
    print(f"  {rid}")


  ANCHOR SET — 20 outputs selected  (seed=127)

▸ Score distribution:
count    20.000000
mean      5.750000
std       2.381397
min       2.000000
25%       4.000000
50%       5.500000
75%       8.000000
max       9.000000

  Min score : 2
  Max score : 9
  Mean score: 5.75

▸ Grounding condition coverage:
grounding_condition
no_context      8
context_only    6
kg_context      6

▸ Company coverage:
company_name
Faik Sönmez    5
Gusto          5
H&M            5
Mavi           5

▸ Prompt type coverage:
prompt_type
concealment    4
irrelevance    4
vagueness      4
misleading     4
overselling    4

▸ Sustainability focus split:
sustainability_focus
False    10
True     10

▸ Selected run IDs:
  Faiksonmez En__concealment__context_only__rep1
  Faiksonmez En__irrelevance__context_only__rep1
  Faiksonmez En__misleading__no_context__rep1
  Faiksonmez En__overselling__no_context__rep1
  Faiksonmez En__vagueness__context_only__rep1
  Gusto En__concealment__kg_context__rep1
  Gusto En__irrel